In [1]:
def kv_kb_per_token(layers=28, kv_heads=2, head_dim=128, dbytes=2):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024


KB_PER_TOKEN = kv_kb_per_token()

BLOCK_TOKENS = 16
BLOCK_KB = BLOCK_TOKENS * KB_PER_TOKEN

print("KB per token:", KB_PER_TOKEN)
print("KB per block:", BLOCK_KB)

KB per token: 28.0
KB per block: 448.0


In [2]:
class SlabAllocator:
    def __init__(self, budget_kb, max_len=4096):
        self.budget_kb = budget_kb
        self.max_len = max_len
        self.used_kb = 0
        self.resident = {}

    def admit(self, seq_id):
        need = self.max_len * KB_PER_TOKEN

        if self.used_kb + need > self.budget_kb:
            return False

        self.used_kb += need
        self.resident[seq_id] = need

        return True

    def complete(self, seq_id):
        self.used_kb -= self.resident.pop(seq_id)

In [3]:
class BlockPoolAllocator:
    def __init__(self, budget_kb, block_kb=BLOCK_KB):
        self.block_kb = block_kb
        self.total_blocks = int(budget_kb // block_kb)
        self.free_blocks = self.total_blocks
        self.block_tables = {}

    def admit(self, seq_id):
        # Every new sequence starts with one block.
        if self.free_blocks < 1:
            return False

        self.free_blocks -= 1
        self.block_tables[seq_id] = 1

        return True

    def grow(self, seq_id, current_len_tokens):
        needed_blocks = -(-current_len_tokens // BLOCK_TOKENS)

        held = self.block_tables[seq_id]

        if needed_blocks > held:
            extra = needed_blocks - held

            if self.free_blocks < extra:
                return False

            self.free_blocks -= extra
            self.block_tables[seq_id] = needed_blocks

        return True

    def complete(self, seq_id):
        self.free_blocks += self.block_tables.pop(seq_id)

In [4]:
import random

random.seed(7)

def make_workload(n_sequences=60, max_len=4096):
    lengths = []

    for _ in range(n_sequences):

        if random.random() < 0.85:
            lengths.append(random.randint(50, 400))
        else:
            lengths.append(random.randint(2000, max_len))

    return lengths


WORKLOAD = make_workload()

print("Number of sequences:", len(WORKLOAD))
print("Mean length:", sum(WORKLOAD) / len(WORKLOAD))
print("Max length:", max(WORKLOAD))

Number of sequences: 60
Mean length: 444.4
Max length: 3763


In [5]:
def simulate_slab(budget_kb, workload):
    alloc = SlabAllocator(budget_kb)

    admitted = 0
    rejected = 0

    for i, length in enumerate(workload):

        if alloc.admit(seq_id=i):
            admitted += 1
        else:
            rejected += 1

    return {
        "peak_concurrent": admitted,
        "admitted": admitted,
        "rejected": rejected
    }

In [6]:
def simulate_blockpool(budget_kb, workload):
    alloc = BlockPoolAllocator(budget_kb)

    admitted = 0
    rejected = 0

    for i, length in enumerate(workload):

        if not alloc.admit(seq_id=i):
            rejected += 1
            continue

        grew = True

        for step_len in range(
            BLOCK_TOKENS,
            length + BLOCK_TOKENS,
            BLOCK_TOKENS
        ):
            if not alloc.grow(
                seq_id=i,
                current_len_tokens=min(step_len, length)
            ):
                grew = False
                break

        if grew:
            admitted += 1
        else:
            alloc.complete(seq_id=i)
            rejected += 1

    return {
        "peak_concurrent": admitted,
        "admitted": admitted,
        "rejected": rejected
    }

In [7]:
BUDGET_KB = 2 * 1024 * 1024

print("Budget:", BUDGET_KB, "KB")

Budget: 2097152 KB


In [8]:
slab_result = simulate_slab(
    BUDGET_KB,
    WORKLOAD
)

blockpool_result = simulate_blockpool(
    BUDGET_KB,
    WORKLOAD
)

print("slab:      ", slab_result)
print("block-pool:", blockpool_result)

slab:       {'peak_concurrent': 18, 'admitted': 18, 'rejected': 42}
block-pool: {'peak_concurrent': 60, 'admitted': 60, 'rejected': 0}


In [9]:
import json

report = {
    "kb_per_token": KB_PER_TOKEN,
    "block_tokens": BLOCK_TOKENS,
    "budget_kb": BUDGET_KB,
    "workload_mean_len": sum(WORKLOAD) / len(WORKLOAD),
    "slab": slab_result,
    "blockpool": blockpool_result,
    "blockpool_advantage": round(
        blockpool_result["peak_concurrent"]
        / slab_result["peak_concurrent"],
        2
    ),
}

with open("kv_sim_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "kb_per_token": 28.0,
  "block_tokens": 16,
  "budget_kb": 2097152,
  "workload_mean_len": 444.4,
  "slab": {
    "peak_concurrent": 18,
    "admitted": 18,
    "rejected": 42
  },
  "blockpool": {
    "peak_concurrent": 60,
    "admitted": 60,
    "rejected": 0
  },
  "blockpool_advantage": 3.33
}


In [10]:
#!/usr/bin/env python3
# Green check for the extra W3D2 lab (the paged KV allocator).
# Run next to kv_sim_report.json:  python verify.py
# Prints exactly one line last: GREEN CHECK: PASS  or  GREEN CHECK: FAIL (<reason>)
# stdlib only. The lab is fully deterministic (random.seed(7)), so this
# verifier recomputes the workload and both simulations with its own reference
# implementation and demands an exact match.
import json, os, random
from typing import NoReturn

KB_PER_TOKEN = 2 * 28 * 2 * 128 * 2 / 1024   # layers 28, kv_heads 2, head_dim 128, fp16
BLOCK_TOKENS = 16
BLOCK_KB = BLOCK_TOKENS * KB_PER_TOKEN
MAX_LEN = 4096
BUDGET_KB = 2 * 1024 * 1024
MIN_ADVANTAGE = 1.5


class _Stop(Exception):
    pass


def _fail(reason) -> NoReturn:
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def make_workload(n_sequences=60, max_len=MAX_LEN):
    random.seed(7)
    lengths = []
    for _ in range(n_sequences):
        if random.random() < 0.85:
            lengths.append(random.randint(50, 400))
        else:
            lengths.append(random.randint(2000, max_len))
    return lengths


def simulate_slab(budget_kb, workload):
    used, admitted, rejected = 0.0, 0, 0
    need = MAX_LEN * KB_PER_TOKEN
    for _ in workload:
        if used + need <= budget_kb:
            used += need
            admitted += 1
        else:
            rejected += 1
    return {"peak_concurrent": admitted, "admitted": admitted, "rejected": rejected}


def simulate_blockpool(budget_kb, workload):
    total_blocks = int(budget_kb // BLOCK_KB)
    free = total_blocks
    admitted = rejected = 0
    for length in workload:
        needed = -(-length // BLOCK_TOKENS)
        if needed < 1:
            needed = 1
        if free >= needed:
            free -= needed
            admitted += 1
        else:
            rejected += 1
    return {"peak_concurrent": admitted, "admitted": admitted, "rejected": rejected}


def check_block(name, got, want):
    for key in ("peak_concurrent", "admitted", "rejected"):
        if got.get(key) != want[key]:
            _fail("%s %s=%r, reference computes %r (deterministic seed; the "
                  "difference is in your allocator or simulation)"
                  % (name, key, got.get(key), want[key]))


def main():
    if not os.path.isfile("kv_sim_report.json"):
        _fail("kv_sim_report.json not found; run Step 6 first")
    try:
        with open("kv_sim_report.json") as f:
            r = json.load(f)
    except json.JSONDecodeError as e:
        _fail("kv_sim_report.json is not valid JSON: %s" % e)

    for key in ("kb_per_token", "block_tokens", "budget_kb", "slab",
                "blockpool", "blockpool_advantage"):
        if key not in r:
            _fail("missing key '%s'" % key)
    if abs(r["kb_per_token"] - KB_PER_TOKEN) > 0.01:
        _fail("kb_per_token=%r, the Qwen2.5-1.5B formula gives %.2f"
              % (r["kb_per_token"], KB_PER_TOKEN))
    if r["block_tokens"] != BLOCK_TOKENS:
        _fail("block_tokens=%r, the lab fixes 16" % r["block_tokens"])
    if r["budget_kb"] != BUDGET_KB:
        _fail("budget_kb=%r, the lab fixes 2 GB (%d KB) so the slab binds"
              % (r["budget_kb"], BUDGET_KB))

    workload = make_workload()
    want_slab = simulate_slab(BUDGET_KB, workload)
    want_pool = simulate_blockpool(BUDGET_KB, workload)
    check_block("slab", r["slab"], want_slab)
    check_block("blockpool", r["blockpool"], want_pool)

    want_adv = round(want_pool["peak_concurrent"] / want_slab["peak_concurrent"], 2)
    if abs(r["blockpool_advantage"] - want_adv) > 0.01:
        _fail("blockpool_advantage=%r, reference computes %s"
              % (r["blockpool_advantage"], want_adv))
    if want_adv < MIN_ADVANTAGE:
        _fail("verifier bug: reference advantage %.2f under the %.1fx bar"
              % (want_adv, MIN_ADVANTAGE))

    print("reference: slab %d resident, block-pool %d resident, advantage %.2fx"
          % (want_slab["peak_concurrent"], want_pool["peak_concurrent"], want_adv))
    print("GREEN CHECK: PASS")


if __name__ == "__main__":
    try:
        main()
    except _Stop:
        raise SystemExit(1)


reference: slab 18 resident, block-pool 60 resident, advantage 3.33x
GREEN CHECK: PASS
